# Comparacion de los Modelos de regresion lineal simple, Rigde, Lasso y Elastic Net 

## Limpieza de datos
Para este caso se comenzara por leer el archivo de los datos para hacer un analisis rapido. 



In [ ]:
import pandas as pd
df = pd.read_csv(r"D:\TODO\Python\files\Dataset\global_climate_energy_2020_2024.csv")
df.head()


,date,country,avg_temperature,humidity,co2_emission,energy_consumption,renewable_share,urban_population,industrial_activity_index,energy_price
0,2020-01-01,Germany,28.29,31.08,212.63,11348.75,14.42,76.39,51.22,83.93
1,2020-01-02,Germany,28.38,37.94,606.05,4166.64,5.63,86.26,78.27,110.40
2,2020-01-03,Germany,28.74,57.67,268.72,4503.80,14.20,75.92,48.96,173.58
3,2020-01-04,Germany,26.66,51.34,167.32,3259.13,13.84,63.15,97.42,89.13
4,2020-01-05,Germany,26.81,65.38,393.89,7023.72,6.93,76.02,81.89,40.60


In [4]:
df.dtypes

date                             str
country                          str
avg_temperature              float64
humidity                     float64
co2_emission                 float64
energy_consumption           float64
renewable_share              float64
urban_population             float64
industrial_activity_index    float64
energy_price                 float64
dtype: object

In [ ]:
#Verificacion que no tenga valores iguales a 0 o vacios
df.isnull().sum()

date                         0
country                      0
avg_temperature              0
humidity                     0
co2_emission                 0
energy_consumption           0
renewable_share              0
urban_population             0
industrial_activity_index    0
energy_price                 0
dtype: int64

In [ ]:
#Tamaño del dataframe
df.shape

(36540, 10)

In [15]:
#Cambio de los nombres de las columnas 
df = df.copy()
new_names =  [
    "Fecha",
    "Pais",
    "Temperatura_promedio_C°",
    "Humedad_%",
    "Emisiones_CO2",
    "Consumo_energia_MWh",
    "Porcentaje_de_Energia_renovable_%",
    "Poblacion_urbana",
    "Indic_de_actividad_industrial",
    "Precio_energia_electrica_€/MWh"
    ]
if len(new_names) != len(df.columns):
    raise ValueError("la cantidad de nombres no coincide con los de la columna")

df.columns = new_names
df.head()


,Fecha,Pais,Temperatura_promedio_C°,Humedad_%,Emisiones_CO2,Consumo_energia_MWh,Porcentaje_de_Energia_renovable_%,Poblacion_urbana,Indic_de_actividad_industrial,Precio_energia_electrica_€/MWh
0,2020-01-01,Germany,28.29,31.08,212.63,11348.75,14.42,76.39,51.22,83.93
1,2020-01-02,Germany,28.38,37.94,606.05,4166.64,5.63,86.26,78.27,110.40
2,2020-01-03,Germany,28.74,57.67,268.72,4503.80,14.20,75.92,48.96,173.58
3,2020-01-04,Germany,26.66,51.34,167.32,3259.13,13.84,63.15,97.42,89.13
4,2020-01-05,Germany,26.81,65.38,393.89,7023.72,6.93,76.02,81.89,40.60


In [18]:
#Cambio de tipo de datos de fecha a datetime64
df["Fecha"] = pd.to_datetime(df["Fecha"],errors="coerce")
df.dtypes

Fecha                                datetime64[us]
Pais                                            str
Temperatura_promedio_C°                     float64
Humedad_%                                   float64
Emisiones_CO2                               float64
Consumo_energia_MWh                         float64
Porcentaje_de_Energia_renovable_%           float64
Poblacion_urbana                            float64
Indic_de_actividad_industrial               float64
Precio_energia_electrica_€/MWh              float64
dtype: object

In [42]:
#correlacion entre las variables
df.corr(numeric_only=True)

,Temperatura_promedio_C°,Humedad_%,Emisiones_CO2,Consumo_energia_MWh,Porcentaje_de_Energia_renovable_%,Poblacion_urbana,Indic_de_actividad_industrial,Precio_energia_electrica_€/MWh
Temperatura_promedio_C°,1.000000,0.004675,0.004647,-0.006845,-0.003454,0.002122,-0.000390,0.000455
Humedad_%,0.004675,1.000000,0.000547,0.003214,0.008300,0.000946,-0.008132,-0.000133
Emisiones_CO2,0.004647,0.000547,1.000000,0.171758,-0.001837,-0.002710,0.002069,0.004401
Consumo_energia_MWh,-0.006845,0.003214,0.171758,1.000000,-0.002070,0.005503,-0.008340,-0.007559
Porcentaje_de_Energia_renovable_%,-0.003454,0.008300,-0.001837,-0.002070,1.000000,-0.006074,-0.004065,0.001826
Poblacion_urbana,0.002122,0.000946,-0.002710,0.005503,-0.006074,1.000000,0.007666,-0.001499
Indic_de_actividad_industrial,-0.000390,-0.008132,0.002069,-0.008340,-0.004065,0.007666,1.000000,0.010577
Precio_energia_electrica_€/MWh,0.000455,-0.000133,0.004401,-0.007559,0.001826,-0.001499,0.010577,1.000000


## Regresion lineal multiple simple 

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
